# ============================================================
# PyTorch Lightning Pipeline (Main Pipeline)
# Used for final experiments and results in the report
# ============================================================

In [1]:
# ============================================================
# Setup
# ============================================================
import os

# Optional: only enable this for CUDA debugging
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import lightning as pl
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from pathlib import Path
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

In [2]:
# ============================================================
# Path configuration
# ============================================================

# Base project directory
BASE_DIR = Path("/p/scratch/training2600/ripp1/Project")

# Train / validation / test directories
TRAIN_DIR = BASE_DIR / "training_data" / "train"
VAL_DIR   = BASE_DIR / "training_data" / "val"
TEST_DIR  = BASE_DIR / "training_data" / "test"

# Check if all required directories exist
assert TRAIN_DIR.exists(), f"Train path not found: {TRAIN_DIR}"
assert VAL_DIR.exists(),   f"Validation path not found: {VAL_DIR}"
assert TEST_DIR.exists(),  f"Test path not found: {TEST_DIR}"

print("Using data from:")
print(f"  Train: {TRAIN_DIR}")
print(f"  Val:   {VAL_DIR}")
print(f"  Test:  {TEST_DIR}")

Using data from:
  Train: /p/scratch/training2600/ripp1/Project/training_data/train
  Val:   /p/scratch/training2600/ripp1/Project/training_data/val
  Test:  /p/scratch/training2600/ripp1/Project/training_data/test


## Custom Dataset Class

In [3]:
# ============================================================
# Utils
# ============================================================

def load_split_data(split_dir, split_name):
    """
    Load patches and labels for one data split.

    Parameters:
    -----------
    split_dir : Path
        Directory of the split (train / val / test)
    split_name : str
        Name of the split ("train", "val", or "test")
        
    Returns:
    --------
    X : np.ndarray
        Patch data with shape (N, H, W, C)
    y : np.ndarray
        Label array with shape (N,)
    """
    patches_path = split_dir / f"patches_{split_name}.npz"
    labels_path = split_dir / f"labels_{split_name}.npy"

    X = np.load(patches_path)["patches"]
    y = np.load(labels_path)

    return X, y


def get_class_distribution(labels):
    """
    Compute class distribution from a label array.

    Parameters:
    -----------
    labels : np.ndarray

    Returns:
    --------
    dict : {class_index: count}, sorted by frequency
    """
    unique, counts = np.unique(labels, return_counts=True)

    distribution = {
        int(k): int(v)
        for k, v in sorted(zip(unique, counts), key=lambda x: -x[1])
    }

    return distribution

In [4]:
# ============================================================
# Label mapping
# ============================================================

# Load original labels from all splits
_, y_train_raw = load_split_data(TRAIN_DIR, "train")
_, y_val_raw = load_split_data(VAL_DIR, "val")
_, y_test_raw = load_split_data(TEST_DIR, "test")

# Build one shared mapping across all splits
all_labels = np.concatenate([y_train_raw, y_val_raw, y_test_raw])
unique_labels = np.unique(all_labels)

label_to_idx = {int(lbl): i for i, lbl in enumerate(unique_labels)}
idx_to_label = {i: int(lbl) for i, lbl in enumerate(unique_labels)}

print("Label mapping (CORINE -> class index):")
for orig, idx in label_to_idx.items():
    print(f"  {orig:2d} -> {idx}")

print(f"\nTotal number of classes: {len(unique_labels)}")

Label mapping (CORINE -> class index):
   2 -> 0
   3 -> 1
   7 -> 2
  11 -> 3
  12 -> 4
  18 -> 5
  20 -> 6
  23 -> 7
  24 -> 8

Total number of classes: 9


In [5]:
# ============================================================
# Dataset
# ============================================================

class CorineDataset(Dataset):
    """
    Dataset for Sentinel-2 patches and CORINE labels.

    Parameters:
    -----------
    data : np.ndarray (N, H, W, C)
    labels : np.ndarray (N,)
    label_mapping : dict
        Mapping from original CORINE labels to contiguous class indices
    """

    def __init__(self, data, labels, label_mapping):
        super().__init__()

        # Convert input data to tensors
        self.data = torch.tensor(data, dtype=torch.float32)

        # Remap labels to contiguous indices: 0 ... num_classes-1
        remapped_labels = np.array(
            [label_mapping[int(lbl)] for lbl in labels],
            dtype=np.int64
        )
        self.labels = torch.tensor(remapped_labels, dtype=torch.long)

        # Convert from (H, W, C) to (C, H, W) for CNN input
        self.data = self.data.permute(0, 3, 1, 2)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]
        y = self.labels[idx]
        return x, y

In [23]:
# ============================================================
# Lightning DataModule
# ============================================================

class CorineDataModule(pl.LightningDataModule):
    """
    Lightning DataModule for Sentinel-2 / CORINE classification.

    This version directly uses the already prepared train / val / test datasets.
    """

    def __init__(self, train_dir: Path, val_dir: Path, test_dir: Path, cfg: dict, label_mapping: dict):
        super().__init__()
        self.train_dir = train_dir
        self.val_dir = val_dir
        self.test_dir = test_dir
        self.cfg = cfg
        self.label_mapping = label_mapping

    def setup(self, stage=None):
        """
        Load data and create datasets.
        """
        # Load split data
        X_train, y_train = load_split_data(self.train_dir, "train")
        X_val, y_val = load_split_data(self.val_dir, "val")
        X_test, y_test = load_split_data(self.test_dir, "test")

        # Create datasets
        self.train_dataset = CorineDataset(X_train, y_train, self.label_mapping)
        self.val_dataset = CorineDataset(X_val, y_val, self.label_mapping)
        self.test_dataset = CorineDataset(X_test, y_test, self.label_mapping)

        # Store remapped labels for diagnostics / optional sampling
        self.train_labels = self.train_dataset.labels
        self.val_labels = self.val_dataset.labels
        self.test_labels = self.test_dataset.labels

    def train_dataloader(self):
        """
        Create training dataloader.

        If sampling_strategy == "weighted_sampler", use weighted sampling.
        If sampling_strategy == "class_weights", use normal shuffling and
        apply class weights later in the loss function.
        """
        sampling_strategy = self.cfg["data"].get("sampling_strategy", "random")

        if sampling_strategy == "weighted_sampler":
            unique_labels, counts = self.train_labels.unique(return_counts=True)
            class_weights = torch.reciprocal(counts.float())

            weights_dict = {
                int(label.item()): float(weight.item())
                for label, weight in zip(unique_labels, class_weights)
            }

            sample_weights = torch.tensor(
                [weights_dict[int(label.item())] for label in self.train_labels],
                dtype=torch.float64,
            )

            sampler = WeightedRandomSampler(
                weights=sample_weights,
                num_samples=len(sample_weights),
                replacement=True,
            )
            shuffle = False

            print("Using weighted random sampler.")
            print("Class weights:", weights_dict)

        else:
            # For "class_weights" or "random"
            sampler = None
            shuffle = True

            if sampling_strategy == "class_weights":
                print("Using class-weighted loss (no sampler in DataLoader).")
            else:
                print("Using random sampling.")

        return DataLoader(
            self.train_dataset,
            batch_size=self.cfg["data"]["batch_size"],
            shuffle=shuffle,
            sampler=sampler,
            num_workers=2,
            pin_memory=True,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.cfg["data"]["batch_size"],
            shuffle=False,
            num_workers=2,
            pin_memory=True,
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_dataset,
            batch_size=self.cfg["data"]["batch_size"],
            shuffle=False,
            num_workers=2,
            pin_memory=True,
        )

In [33]:
# ============================================================
# Configuration
# ============================================================

config = {
    "data": {
        "batch_size": 128,
        "sampling_strategy": "class_weights",   #options: class_weights, weighted_sample, random
    },
    "model": {
        "in_channels": 4,
        "patch_size": 3,
    },
    "training": {
        "epochs": 150,
        "lr": 1e-4,
    },
}

In [34]:
# ============================================================
# Create DataModule
# ============================================================

datamodule = CorineDataModule(
    train_dir=TRAIN_DIR,
    val_dir=VAL_DIR,
    test_dir=TEST_DIR,
    cfg=config,
    label_mapping=label_to_idx,
)

# Prepare datasets
datamodule.setup()

print("Datasets loaded successfully:")
print(f"  Train samples: {len(datamodule.train_dataset)}")
print(f"  Val samples:   {len(datamodule.val_dataset)}")
print(f"  Test samples:  {len(datamodule.test_dataset)}")

Datasets loaded successfully:
  Train samples: 4000
  Val samples:   1000
  Test samples:  1000


In [35]:
# ============================================================
# Quick DataLoader sanity check
# ============================================================

train_loader = datamodule.train_dataloader()
val_loader = datamodule.val_dataloader()
test_loader = datamodule.test_dataloader()

xb, yb = next(iter(train_loader))

print("Quick data check:")
print(f"  Batch X shape: {xb.shape} (dtype={xb.dtype})")
print(f"  Batch y shape: {yb.shape} (dtype={yb.dtype})")
print(f"  y range:       [{yb.min().item()}, {yb.max().item()}]")
print(f"  Unique classes in batch: {sorted(torch.unique(yb).tolist())}")

Using class-weighted loss (no sampler in DataLoader).
Quick data check:
  Batch X shape: torch.Size([128, 4, 3, 3]) (dtype=torch.float32)
  Batch y shape: torch.Size([128]) (dtype=torch.int64)
  y range:       [0, 8]
  Unique classes in batch: [0, 3, 4, 5, 7, 8]


In [36]:
# ============================================================
# Model
# ============================================================

class ConvNet(pl.LightningModule):
    """
    Simple CNN classifier for Sentinel-2 patch classification.

    Expected input shape:
        (B, C, H, W)

    For this project:
        C = 4
        H = W = 3
    """

    def __init__(
        self,
        num_classes,
        in_channels=4,
        patch_size=3,
        lr=3e-4,
        max_epochs=100,
        class_weights=None,
    ):
        super().__init__()

        self.lr = lr
        self.in_channels = in_channels
        self.patch_size = patch_size
        self.num_classes = num_classes
        self.max_epochs = max_epochs

        # Feature extraction layers
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d((1, 1)),
        )

        # Classification head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, num_classes),
        )

        if class_weights is not None:
            self.register_buffer("class_weights", class_weights.float())
        else:
            self.class_weights = None

    def forward(self, x):
        x = x.float()
        x = self.features(x)
        return self.classifier(x)

    def _loss(self, logits, y):
        return F.cross_entropy(logits, y, weight=self.class_weights)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self._loss(logits, y)
        acc = (logits.argmax(1) == y).float().mean()

        self.log("train_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("train_acc", acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self._loss(logits, y)
        acc = (logits.argmax(1) == y).float().mean()

        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("val_acc", acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self._loss(logits, y)
        acc = (logits.argmax(1) == y).float().mean()

        self.log("test_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("test_acc", acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=self.max_epochs
        )
        return {"optimizer": optimizer, "lr_scheduler": scheduler}

In [37]:
# ============================================================
# Model setup
# ============================================================

# Number of classes
num_classes = len(torch.unique(datamodule.train_labels))

# ============================================================
# Compute class weights (sqrt version = more stable)
# ============================================================

unique_labels, counts = datamodule.train_labels.unique(return_counts=True)

# inverse sqrt weighting (weniger extrem als 1/count)
class_weights = torch.zeros(num_classes, dtype=torch.float32)

for label, count in zip(unique_labels, counts):
    class_weights[int(label.item())] = 1.0 / torch.sqrt(count.float())

# normalize (wichtig!)
class_weights = class_weights / class_weights.mean()

print("Class weights:")
for cls_idx, weight in enumerate(class_weights):
    print(f"  Class {cls_idx:2d} -> weight {weight.item():.3f}")

# only use weights if strategy = class_weights
if config["data"]["sampling_strategy"] == "class_weights":
    class_weights_for_loss = class_weights
else:
    class_weights_for_loss = None


# ============================================================
# Create model
# ============================================================

model = ConvNet(
    num_classes=num_classes,
    in_channels=config["model"]["in_channels"],
    patch_size=config["model"]["patch_size"],
    lr=config["training"]["lr"],
    max_epochs=config["training"]["epochs"],
    class_weights=class_weights_for_loss,
)

print("\nModel created successfully.")
print(f"  Number of classes: {num_classes}")
print(f"  Learning rate:     {config['training']['lr']}")

Class weights:
  Class  0 -> weight 0.416
  Class  1 -> weight 1.295
  Class  2 -> weight 1.154
  Class  3 -> weight 1.504
  Class  4 -> weight 0.609
  Class  5 -> weight 0.204
  Class  6 -> weight 2.543
  Class  7 -> weight 0.343
  Class  8 -> weight 0.933

Model created successfully.
  Number of classes: 9
  Learning rate:     0.0001


In [38]:
# ============================================================
# Training
# ============================================================

print("\n" + "=" * 70)
print("STARTING TRAINING")
print("=" * 70)
print(f"Sampling strategy: {config['data']['sampling_strategy']}")
print(f"Max epochs:        {config['training']['epochs']}")
print(f"Batch size:        {config['data']['batch_size']}")
print("=" * 70 + "\n")

trainer = pl.Trainer(
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    max_epochs=config["training"]["epochs"],
    gradient_clip_val=1.0,
    log_every_n_steps=1,
    enable_progress_bar=True,
)

trainer.fit(model, datamodule=datamodule)

print("\n" + "=" * 70)
print("TESTING")
print("=" * 70)

trainer.test(model, datamodule=datamodule)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/p/project1/training2600/ripp1/envs/ml_eo_course/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /p/project1/training2600/ripp1/Project/lightning_logs/version_14673802/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3     ]



STARTING TRAINING
Sampling strategy: class_weights
Max epochs:        150
Batch size:        128



┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ features   │ Sequential │ 76.6 K │ train │     0 │
│ 1 │ classifier │ Sequential │  8.8 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 85.4 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 85.4 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 14                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

SLURM auto-requeueing enabled. Setting signal handlers.


...

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3     ]
SLURM auto-requeueing enabled. Setting signal handlers.



TESTING


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.4580000042915344     │
│         test_loss         │     1.467682957649231     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 1.467682957649231, 'test_acc': 0.4580000042915344}]

In [42]:
# ============================================================
# FULL EVALUATION + SAVING
# ============================================================

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ------------------------------------------------------------
# OUTPUT FOLDER 
# ------------------------------------------------------------
EVAL_DIR = BASE_DIR / "results" /"evaluation"/ "class_weights"   
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Saving evaluation outputs to: {EVAL_DIR}")


# ------------------------------------------------------------
# FUNCTIONS
# ------------------------------------------------------------
def collect_preds_targets(model, dataloader):
    model.eval()
    device = next(model.parameters()).device

    preds_all = []
    targets_all = []

    with torch.no_grad():
        for xb, yb in dataloader:
            xb = xb.to(device)
            logits = model(xb)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            targets = yb.cpu().numpy()

            preds_all.append(preds)
            targets_all.append(targets)

    y_pred = np.concatenate(preds_all).astype(np.int64)
    y_true = np.concatenate(targets_all).astype(np.int64)

    return y_true, y_pred


def compute_confusion_matrix(y_true, y_pred):
    labels = np.unique(np.concatenate([y_true, y_pred]))
    label_to_idx = {int(lbl): i for i, lbl in enumerate(labels)}

    cm = np.zeros((len(labels), len(labels)), dtype=np.int64)

    for t, p in zip(y_true, y_pred):
        cm[label_to_idx[int(t)], label_to_idx[int(p)]] += 1

    return labels, cm


def compute_imbalanced_metrics(cm):
    tp = np.diag(cm).astype(np.float64)
    support = cm.sum(axis=1).astype(np.float64)
    pred_count = cm.sum(axis=0).astype(np.float64)

    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    precision = np.divide(tp, pred_count, out=np.zeros_like(tp), where=pred_count > 0)

    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(tp),
        where=(precision + recall) > 0,
    )

    overall_acc = tp.sum() / cm.sum()
    balanced_acc = recall.mean()
    macro_f1 = f1.mean()

    return overall_acc, balanced_acc, macro_f1, recall, precision, f1


def save_metrics(results, split_name, output_dir):
    filepath = output_dir / f"{split_name}_metrics.txt"

    with open(filepath, "w") as f:
        f.write(f"{split_name.upper()} METRICS\n")
        f.write("=" * 40 + "\n")
        f.write(f"Overall accuracy:  {results['overall_acc']:.4f}\n")
        f.write(f"Balanced accuracy:{results['balanced_acc']:.4f}\n")
        f.write(f"Macro-F1:         {results['macro_f1']:.4f}\n")
        f.write(f"Majority baseline:{results['majority_baseline']:.4f}\n")

    print(f"Saved metrics to: {filepath}")


def save_text_report(filepath, split_name, majority_baseline, overall_acc, balanced_acc, macro_f1,
                     labels, cm, recall, precision, f1, y_pred):

    pred_vals, pred_cnts = np.unique(y_pred, return_counts=True)
    pred_order = np.argsort(pred_cnts)[::-1]

    with open(filepath, "w") as f:
        f.write("=" * 70 + "\n")
        f.write(f"{split_name.upper()} DIAGNOSTICS\n")
        f.write("=" * 70 + "\n")
        f.write(f"Majority-class baseline acc: {majority_baseline:.4f}\n")
        f.write(f"Overall accuracy:            {overall_acc:.4f}\n")
        f.write(f"Balanced accuracy:           {balanced_acc:.4f}\n")
        f.write(f"Macro-F1:                    {macro_f1:.4f}\n\n")

        f.write("Predicted class distribution:\n")
        for cls, c in zip(pred_vals[pred_order], pred_cnts[pred_order]):
            f.write(f"  class {int(cls):2d}: {int(c)}\n")

        f.write("\nPer-class metrics:\n")
        f.write("class | support | recall | precision | f1\n")
        for i, cls in enumerate(labels):
            sup = int(cm[i].sum())
            f.write(
                f"{int(cls):5d} | {sup:7d} | {recall[i]:6.3f} | {precision[i]:9.3f} | {f1[i]:6.3f}\n"
            )

        f.write("\nConfusion matrix:\n")
        f.write(np.array2string(cm))


def save_confusion_matrix_plot(cm, labels, split_name, save_path):
    plt.close("all")
    plt.figure(figsize=(8,6))

    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels)

    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Confusion Matrix ({split_name})")

    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.show()


def evaluate_and_save(model, dataloader, split_name, output_dir):
    y_true, y_pred = collect_preds_targets(model, dataloader)
    labels, cm = compute_confusion_matrix(y_true, y_pred)

    overall_acc, balanced_acc, macro_f1, recall, precision, f1 = compute_imbalanced_metrics(cm)

    vals, cnts = np.unique(y_true, return_counts=True)
    majority_baseline = cnts.max() / cnts.sum()

    print("\n" + "="*70)
    print(f"{split_name.upper()} RESULTS")
    print("="*70)
    print(f"Accuracy:          {overall_acc:.4f}")
    print(f"Balanced Accuracy: {balanced_acc:.4f}")
    print(f"Macro-F1:          {macro_f1:.4f}")

    results = {
        "overall_acc": overall_acc,
        "balanced_acc": balanced_acc,
        "macro_f1": macro_f1,
        "majority_baseline": majority_baseline,
        "cm": cm,
        "labels": labels
    }

    # Save everything
    save_metrics(results, split_name, output_dir)

    save_text_report(
        output_dir / f"{split_name}_diagnostics.txt",
        split_name,
        majority_baseline,
        overall_acc,
        balanced_acc,
        macro_f1,
        labels,
        cm,
        recall,
        precision,
        f1,
        y_pred
    )

    np.save(output_dir / f"{split_name}_confusion_matrix.npy", cm)

    save_confusion_matrix_plot(
        cm,
        labels,
        split_name,
        output_dir / f"{split_name}_confusion_matrix.png"
    )

    return results


# ------------------------------------------------------------
# RUN EVALUATION
# ------------------------------------------------------------
val_results = evaluate_and_save(
    model,
    datamodule.val_dataloader(),
    "validation",
    EVAL_DIR
)

test_results = evaluate_and_save(
    model,
    datamodule.test_dataloader(),
    "test",
    EVAL_DIR
)

Saving evaluation outputs to: /p/scratch/training2600/ripp1/Project/results/evaluation/class_weights

VALIDATION RESULTS
Accuracy:          0.3590
Balanced Accuracy: 0.2125
Macro-F1:          0.1804
Saved metrics to: /p/scratch/training2600/ripp1/Project/results/evaluation/class_weights/validation_metrics.txt

TEST RESULTS
Accuracy:          0.4580
Balanced Accuracy: 0.2169
Macro-F1:          0.2119
Saved metrics to: /p/scratch/training2600/ripp1/Project/results/evaluation/class_weights/test_metrics.txt
